# Homework 3 — Machine Learning for Classification

ML Zoomcamp 2026. Predict whether a lead `converted` using logistic regression.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

## Data preparation — fill missing values

In [3]:
categorical = list(df.select_dtypes(exclude='number').columns)
numerical = [c for c in df.select_dtypes('number').columns if c != 'converted']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
categorical, numerical

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'])

## Q1 — mode of `industry`

In [4]:
df.industry.value_counts().head()

industry
technology    1173
retail         925
healthcare     840
finance        720
education      639
Name: count, dtype: int64

## Q2 — correlation matrix of numerical features

In [5]:
corr = df[numerical].corr()
pairs = [('interaction_count', 'lead_score'), ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'), ('annual_income', 'interaction_count')]
display(corr.round(4))
for a, b in pairs:
    print(a, b, round(corr.loc[a, b], 4))

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.0000,0.1613,0.1228,0.2295
number_of_courses_viewed,0.1613,1.0000,0.7216,0.7572
interaction_count,0.1228,0.7216,1.0000,0.9157
lead_score,0.2295,0.7572,0.9157,1.0000


interaction_count lead_score 0.9157
number_of_courses_viewed lead_score 0.7572
number_of_courses_viewed interaction_count 0.7216
annual_income interaction_count 0.1228


## Split

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.pop('converted').values
y_val = df_val.pop('converted').values
y_test = df_test.pop('converted').values
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Q3 — mutual information with `converted` (training set)

In [7]:
mi = {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Q4 — logistic regression with one-hot encoding

In [8]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

features = categorical + numerical
base_acc = train_and_score(features)
print('accuracy:', round(base_acc, 2), f'({base_acc:.4f})')

accuracy: 0.65 (0.6450)


## Q5 — feature elimination

In [9]:
diffs = {f: base_acc - train_and_score([x for x in features if x != f]) for f in features}
pd.Series(diffs).sort_values()

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

## Q6 — regularization

In [10]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_score(features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
